# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is a **scoring task**. One row represents one webpage, and each webpage will receive a content opportunity score showing how strongly it should be considered for a refresh. The score can then be used to rank pages from highest to lowest priority so the content team can review the strongest opportunities first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My ideal target is whether a webpage gains search impressions after it is refreshed, measured in a later time period. This would be an observed outcome rather than a manually defined rule.

The starter dataset does not contain confirmed before-and-after refresh results, so I will initially use future content decline as a proxy. The proxy would indicate whether a page declines in a later observation window. This helps identify pages that may need attention, but it does not prove that refreshing the page will cause recovery.

For the starter exercise, the target column could be represented as `future_decline`, where:

- `1` means the page showed an observed decline in the later period.
- `0` means the page did not show an observed decline in the later period.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is **Precision@100**. It measures how many of the 100 webpages with the highest predicted decline scores actually show a decline in the later observation period.

I would consider a Precision@100 of **0.80 or higher** to be good. This means that at least 80 of the top 100 pages identified by the model truly declined. This metric fits the real action because the content team has limited time and needs the highest-priority recommendations to be reliable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

My unit of analysis is **one pseudonymized webpage or content item per row**.

For my content opportunity scoring lane, I will use pages with at least 100 impressions during the previous 90 days. This removes pages with almost no search exposure while keeping pages that the content team could realistically evaluate.

Each row contains signals such as impressions, clicks, CTR, average position, content age, and time since the last update.

The `future_decline` column is only a target sketch. It would be filled later using an observed future period. I will not invent its values in the current dataset.

In [ ]:
import pandas as pd

data_url = (
    "https://raw.githubusercontent.com/"
    "ZubairQazzi/flyrank-ml-internship-zubair/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

lane_df = df.loc[
    df["impressions_90d"] >= 100,
    [
        "content_id",
        "client_id",
        "content_type",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update",
        "impressions_prev_30d",
        "impressions_last_30d",
    ],
].copy()

# Placeholder for an outcome observed in a future period
lane_df["future_decline"] = pd.Series(
    pd.NA,
    index=lane_df.index,
    dtype="Int64"
)

print("Rows in full dataset:", len(df))
print("Rows in my lane slice:", len(lane_df))

lane_df.head()

Rows in full dataset: 30000
Rows in my lane slice: 22006


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,impressions_prev_30d,impressions_last_30d,future_decline
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20,987,578,<NA>
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25,5915,2501,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,6089,2382,<NA>
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22,4206,3626,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14,6452,4211,<NA>


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as “prioritize every page whose last-30-day impressions are lower than its previous-30-day impressions” treats every decline as equally important.

In reality, refresh priority depends on several connected signals. A page may be declining but have very low visibility, while another page may have high historical impressions, a useful search position, weak CTR, old content, and a long time since its last update.

A scoring model can combine these signals and learn which combinations are most useful for prioritization. It can therefore produce a more practical review queue than one yes-or-no condition.

A wrong high score may waste an editor's time, while a missed high-value page may allow an important opportunity to continue declining. The model will be used only for decision support, and it should later be compared with a simple rule baseline before claiming that it performs better.

In [ ]:
rule_check = lane_df.loc[lane_df["avg_position"] > 0].copy()

rule_check["simple_decline_rule"] = (
    rule_check["impressions_last_30d"]
    < rule_check["impressions_prev_30d"]
)

rule_summary = (
    rule_check.groupby("simple_decline_rule")[
        [
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
        ]
    ]
    .median()
    .round(2)
)

rule_summary

,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update
simple_decline_rule,,,,,
False,1325.0,0.16,14.8,300.0,22.0
True,1812.0,0.14,11.6,228.0,22.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.